# Chest X-ray Two-Stage Triage on Kaggle
Turn on **GPU** and **Internet** in the right panel before running. See `docs/GUIDE.md` for an explanation of each step.

In [ ]:
!nvidia-smi
!ls /kaggle/input

## 1. Copy the code and install dependencies
Set `CODE_DIR` to match your code dataset name.

In [ ]:
CODE_DIR = '/kaggle/input/cxr-triage-code/chest-xray-triage'
!cp -r {CODE_DIR} /kaggle/working/
%cd /kaggle/working/chest-xray-triage
!pip install -q ensemble-boxes ultralytics pydicom

## 1b. Smoke test (2–3 minutes)
Runs every stage of the pipeline on tiny synthetic data with tiny models: data, both trainings, evaluation, error analysis, external validation, benchmarks, ONNX export, impact, plots, report and the API with Grad-CAM. If this passes, the environment is ready for the real run. It checks that the code runs, not model quality.

In [ ]:
!pip install -q onnx onnxruntime
!python scripts/smoke_test.py

## 2. Choose a run profile and create a Kaggle config
- `quick`: 30% stratified subset, fewer epochs, 640 px detector. About 1-2 hours. Results are labeled *quick* everywhere.
- `default`: full dataset and training budget. Several hours; use **Save Version → Save & Run All**.

Start with `quick` to get real, end-to-end numbers today, then run `default`.

In [ ]:
import yaml
PROFILE = 'quick'          # 'quick' or 'default'
cfg = yaml.safe_load(open(f'configs/{PROFILE}.yaml'))
COMP = '/kaggle/input/vinbigdata-chest-xray-abnormalities-detection'
cfg['data']['raw_csv']   = f'{COMP}/train.csv'
cfg['data']['image_dir'] = '/kaggle/input/<PNG-DATASET>/train'   # <-- edit
cfg['data']['meta_csv']  = 'data/raw/train_meta.csv'             # built in the next cell (sizes + sex/age)
cfg['data']['copy_images'] = True   # safer than symlinks on Kaggle
cfg['classifier']['num_workers'] = 2
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(open('configs/kaggle.yaml').read())

### Build the meta file from DICOM headers
Reads only headers (a few minutes). Gives original image sizes for correct boxes, plus patient sex and age for the subgroup analysis.

In [ ]:
!python -m src.make_meta --dicom_dir {COMP}/train --out data/raw/train_meta.csv
import pandas as pd; m = pd.read_csv('data/raw/train_meta.csv'); print(m.head()); print(m[['sex']].value_counts(dropna=False)); print(m['age'].describe())

## 3. Prepare the data

In [ ]:
!python -m src.data --config configs/kaggle.yaml

### Sanity check: draw the fused boxes on a few images
If the boxes are in the wrong place (e.g. pushed into a corner), the meta file is wrong. **Do not train until this looks right.**

In [ ]:
import matplotlib.patches as pa
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.data import CLASS_NAMES

lab = pd.read_csv('data/processed/labels.csv'); bx = pd.read_csv('data/processed/boxes.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, iid in zip(axes, bx.image_id.drop_duplicates().sample(3, random_state=1)):
    img = Image.open(lab.set_index('image_id').at[iid, 'path']); W, H = img.size
    ax.imshow(img, cmap='gray'); ax.axis('off')
    for r in bx[bx.image_id == iid].itertuples():
        ax.add_patch(pa.Rectangle(((r.xc-r.w/2)*W, (r.yc-r.h/2)*H), r.w*W, r.h*H, fill=False, color='red', lw=2))
        ax.text((r.xc-r.w/2)*W, (r.yc-r.h/2)*H, CLASS_NAMES[r.class_id], color='yellow', fontsize=8)
plt.show()

## 4. Train the classifier (stage 1)

In [ ]:
!python -m src.train_classifier --config configs/kaggle.yaml

## 5. Train YOLOv8 (stage 2)

In [ ]:
!python -m src.train_detector --config configs/kaggle.yaml

## 6. Evaluate
This also writes `gate_threshold_from_val` and `temperature` back into the config.

In [ ]:
!python -m src.evaluate --config configs/kaggle.yaml
import json

ev = json.load(open('runs/eval.json'))['classifier']
cfg['pipeline']['gate_threshold'] = ev['gate_threshold_from_val']
cfg['pipeline']['temperature'] = ev['temperature']
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(f"gate_threshold = {ev['gate_threshold_from_val']:.4f} | T = {ev['temperature']:.3f} | "
      f"ECE {ev['test_ece_before_calibration']:.3f} -> {ev['test_ece_after_calibration']:.3f}")

## 6b. Error analysis: where does the gate miss?
Breaks down abnormal test studies that the gate cleared by finding class, and lists the lowest-scoring ones for review.

In [ ]:
!python -m src.error_analysis --config configs/kaggle.yaml
import pandas as pd
pd.read_csv('runs/gate_false_negatives.csv').head(15)

## 6c. (Optional) External validation on NIH ChestX-ray14 (US)
1. **Add Input** → search for the NIH Chest X-rays dataset and add it.
2. Check the paths with `ls`, then run the cell. The gate runs unchanged (same weights, T and τ) on a stratified sample of frontal views.
3. Takes roughly 10–20 minutes for 3,000 studies on a T4. Skip this cell if you have not added the dataset.

In [ ]:
import os, yaml
NIH = '/kaggle/input/data'                       # <-- check with: !ls /kaggle/input
if os.path.exists(f'{NIH}/Data_Entry_2017.csv'):
    cfg['external'].update(nih_csv=f'{NIH}/Data_Entry_2017.csv', nih_images=NIH)
    yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
    !python -m src.external_eval --config configs/kaggle.yaml
else:
    print('NIH dataset not attached; skipping external validation.')

## 7. Benchmark latency: PyTorch, then ONNX

In [ ]:
!python -m src.benchmark --config configs/kaggle.yaml --half
!pip install -q onnx onnxruntime-gpu
!python -m src.export_onnx --config configs/kaggle.yaml
!python -m src.benchmark --config configs/kaggle.yaml --backend onnx

## 8. Visualize predictions on test images

In [ ]:
from src.pipeline import TwoStagePipeline

pipe = TwoStagePipeline(cfg['classifier']['out'], cfg['detector']['best'],
                        gate_threshold=cfg['pipeline']['gate_threshold'],
                        temperature=cfg['pipeline']['temperature'], det_imgsz=cfg['detector']['img_size'])
te = lab[lab.split == 'test'].sample(3, random_state=3)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, te.itertuples()):
    img = Image.open(r.path); out = pipe(img)
    ax.imshow(img, cmap='gray'); ax.axis('off')
    ax.set_title(f"GT abnormal={r.abnormal} | P={out['abnormal_prob']:.2f} | {out['timings']['total_ms']:.0f}ms")
    for d in out['detections']:
        x1, y1, x2, y2 = d['box_xyxy']
        ax.add_patch(pa.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='lime', lw=2))
        ax.text(x1, y1, f"{d['class_name']} {d['confidence']:.2f}", color='lime', fontsize=8)
plt.show()

## 8b. Grad-CAM: what drove the gate's decision
Check that the heat lands on the lungs and heart, not on text markers or image borders. If it doesn't, that's shortcut learning, and worth mentioning in interviews.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, lab[(lab.split == 'test') & (lab.abnormal == 1)].sample(3, random_state=5).itertuples()):
    heat, p = pipe.explain(Image.open(r.path))
    ax.imshow(heat); ax.axis('off'); ax.set_title(f"P(abnormal) = {p:.2f}")
plt.show()

## 8c. Impact, figures and README results
This estimates operational impact, draws the figures and fills the README and site Results with your real numbers.

In [ ]:
!python -m src.impact --config configs/kaggle.yaml   # GPU hours, cost, missed studies
!python -m src.plots --config configs/kaggle.yaml
!python -m src.report   # writes real numbers + figures into README.md
from IPython.display import Image as Show, display
import os
for f in ['roc', 'reliability', 'per_class_ap', 'training']:
    path = f'docs/figures/{f}.png'
    display(Show(path, width=420)) if os.path.exists(path) else print('not produced:', path)

## 9. Bundle everything for download
The zip holds weights, results, figures, the updated `README.md` and `site/results.json`. Copy `README.md`, `docs/figures/` and `site/results.json` into your local repo and push. Weights stay out of git.

In [ ]:
!mkdir -p export && cp {cfg['classifier']['out']} export/classifier.pt && cp {cfg['detector']['best']} export/detector.pt
!cp {cfg['export']['dir']}/*.onnx {cfg['export']['dir']}/*.json export/ 2>/dev/null
!mkdir -p export/docs && cp -r docs/figures export/docs/ 2>/dev/null; cp README.md export/; mkdir -p export/site && cp site/results.json export/site/ 2>/dev/null
!cp runs/eval.json runs/benchmark_*.json runs/impact.json runs/gate_misses.json runs/gate_false_negatives.csv runs/external_nih.json configs/kaggle.yaml export/ 2>/dev/null
!cd export && zip -qr ../cxr_export.zip * && ls -lh ../cxr_export.zip